# Notebook 6: Complete DeepSeek Block - JAX

In [ ]:
import jax.numpy as jnp
from flax import linen as nn
from jax import random

class RMSNorm(nn.Module):
    """RMSNorm in Flax"""
    hidden_size: int
    eps: float = 1e-6
    
    def setup(self):
        self.weight = self.param('weight', lambda rng, shape: jnp.ones(shape), self.hidden_size)
    
    def __call__(self, x):
        rms = jnp.sqrt(jnp.mean(x ** 2, axis=-1, keepdims=True) + self.eps)
        return x / rms * self.weight

class DeepSeekBlock(nn.Module):
    """Complete block in Flax"""
    config: dict
    
    def setup(self):
        cfg = self.config
        self.attn_norm = RMSNorm(cfg['hidden_size'])
        self.ffn_norm = RMSNorm(cfg['hidden_size'])
        
        # Import from previous notebooks (simplified here)
        self.attention = nn.Dense(cfg['hidden_size'])  # Simplified
        self.moe = nn.Dense(cfg['hidden_size'])  # Simplified
    
    def __call__(self, x):
        # Pre-LN
        attn_input = self.attn_norm(x)
        attn_output = self.attention(attn_input)
        x = x + attn_output
        
        ffn_input = self.ffn_norm(x)
        ffn_output = self.moe(ffn_input)
        x = x + ffn_output
        
        return x

config = {
    'hidden_size': 512,
    'num_experts': 8,
    'num_active_experts': 2,
    'intermediate_size': 1376
}

block = DeepSeekBlock(config)
rng = random.PRNGKey(42)
x = random.normal(rng, (2, 16, 512))
params = block.init(rng, x)
output = block.apply(params, x)
print(f"Block input: {x.shape}")
print(f"Block output: {output.shape}")

## Verification
1. ✅ Can implement RMSNorm in Flax
2. ✅ Can assemble complete block
3. ✅ Understand Pre-LN architecture